In [2]:
from pathlib import Path
import tempfile

import numpy as np
import pandas as pd

from noisyvalue import NoisyFloat, NoisyInt, load, save, sample_noisy_values
from noisyvalue.pandas import NoisyFloatArray, NoisyIntArray

# -----------------------------------------------------------------------------
# 1) Build a DataFrame with noisy columns using the pandas extension arrays.
# -----------------------------------------------------------------------------
regions = ["North", "South", "East", "West"]

# Counts with binomial uncertainty.
visit_counts = [
    NoisyInt.binomial(120, 0.45, rng=1),
    NoisyInt.binomial(90, 0.55, rng=2),
    NoisyInt.binomial(110, 0.50, rng=3),
    NoisyInt.binomial(80, 0.60, rng=4),
]

# Revenue with normal uncertainty.
revenues = [
    NoisyFloat.normal(loc=120.0, scale=10.0, rng=11),
    NoisyFloat.normal(loc=95.0, scale=10.0, rng=12),
    NoisyFloat.normal(loc=130.0, scale=10.0, rng=13),
    NoisyFloat.normal(loc=80.0, scale=10.0, rng=14),
]

df = pd.DataFrame(
    {
        "region": regions,
        "visits": pd.Series(NoisyIntArray._from_sequence(visit_counts)),
        "revenue": pd.Series(NoisyFloatArray._from_sequence(revenues)),
    }
)

print("Original table:")
print(df)
print("\nColumn dtypes:")
print(df.dtypes)

# -----------------------------------------------------------------------------
# 2) Bulk operations through pandas (vectorized arithmetic/comparison/reduce).
# -----------------------------------------------------------------------------
# Vectorized arithmetic between noisy columns.
df["revenue_per_visit"] = df["revenue"] / df["visits"]

# Vectorized comparisons produce nullable boolean masks.
high_revenue_mask = df["revenue"] > 100.0
high_traffic_mask = df["visits"] >= 50

# Reductions keep uncertainty as noisy scalars.
total_revenue = df["revenue"].sum()
mean_revenue_per_visit = df["revenue_per_visit"].mean()

print("\nBulk operation results:")
print(df[["region", "visits", "revenue", "revenue_per_visit"]])
print("\nHigh revenue mask:", high_revenue_mask.tolist())
print("High traffic mask:", high_traffic_mask.tolist())
print("Regions with high revenue:", df.loc[high_revenue_mask, "region"].tolist())
print("Total revenue (noisy):", total_revenue)
print("Mean revenue/visit (noisy):", mean_revenue_per_visit)

# -----------------------------------------------------------------------------
# 3) Exercise IO: save and load DataFrame, then verify behavior still works.
# -----------------------------------------------------------------------------
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "noisy_dataframe_demo.json"
    save(path, df)
    loaded_df = load(path)

print("\nLoaded table:")
print(loaded_df)
print("\nLoaded dtypes:")
print(loaded_df.dtypes)

# Run the same bulk ops after loading.
loaded_total_revenue = loaded_df["revenue"].sum()
loaded_mean_revenue_per_visit = (loaded_df["revenue"] / loaded_df["visits"]).mean()
print("\nLoaded total revenue (noisy):", loaded_total_revenue)
print("Loaded mean revenue/visit (noisy):", loaded_mean_revenue_per_visit)

# Optional sanity check: loaded values preserve the same posterior distribution.
orig_draws, loaded_draws = sample_noisy_values(
    df["revenue"].array[0], loaded_df["revenue"].array[0], n=5000, rng=123
)
orig_mean, orig_std = np.mean(orig_draws.draws), np.std(orig_draws.draws)
loaded_mean, loaded_std = np.mean(loaded_draws.draws), np.std(loaded_draws.draws)
print(
    "\nFirst-row revenue posterior (sample mean/std): "
    f"original=({orig_mean:.3f}, {orig_std:.3f}), "
    f"loaded=({loaded_mean:.3f}, {loaded_std:.3f})"
)

Original table:
  region visits              revenue
0  North    ~48  ~120.34192767253184
1  South    ~47   ~94.93173220134477
2   East    ~54  ~148.26756559957423
3   West    ~60   ~86.95519770038169

Column dtypes:
region            str
visits       noisyint
revenue    noisyfloat
dtype: object

Bulk operation results:
  region visits              revenue    revenue_per_visit
0  North    ~48  ~120.34192767253184  ~2.5071234931777466
1  South    ~47   ~94.93173220134477   ~2.019824089390314
2   East    ~54  ~148.26756559957423  ~2.7456956592513744
3   West    ~60   ~86.95519770038169  ~1.4492532950063615

High revenue mask: [True, False, True, False]
High traffic mask: [False, False, True, True]
Regions with high revenue: ['North', 'East']
Total revenue (noisy): ~450.4964231738325
Mean revenue/visit (noisy): ~2.1804741342064493

Loaded table:
  region visits              revenue    revenue_per_visit
0  North    ~48  ~120.34192767253184  ~2.5071234931777466
1  South    ~47   ~94.9317322